In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import FloatSlider, IntSlider, VBox, HBox, HTML, interactive_output, Layout, GridBox
from IPython.display import display

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.40;
    width:1050px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#12388c;
    margin-bottom:8px;
">
Design of an AR(1) Random Process from Statistical Specifications
</div>

<div style="margin-bottom:4px;">
<b>Goal:</b> choose the parameters of y[n] = αy[n−1] + βw[n] so that the output has prescribed variance and lag-1 correlation.
</div>

<div style="margin-bottom:4px;">
The desired specifications are Rᵧᵧ[0] = σ² and Rᵧᵧ[1] = ρσ².
</div>

<div style="margin-bottom:4px;">
For unit-variance white-noise input, the required parameters are α = ρ and β = σ√(1−ρ²).
</div>

<div>
<b>This notebook:</b> compares the theoretical and estimated autocorrelation and PSD of the generated AR(1) process.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='165px')

rho_slider = FloatSlider(min=0.0, max=0.95, step=0.05, value=0.75, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)
sigma_slider = FloatSlider(min=0.5, max=3.0, step=0.1, value=1.5, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)
N_slider = IntSlider(min=500, max=5000, step=500, value=2000, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)
lag_slider = IntSlider(min=10, max=80, step=5, value=30, description=' ', continuous_update=True, readout=False, style=slider_style, layout=slider_layout)

# ============================================================
# CURRENT VALUE LABELS
# ============================================================

rho_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.75</div>')
sigma_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.5</div>')
N_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">2000</div>')
lag_value = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">30</div>')

# ============================================================
# UPDATE VALUE LABELS
# ============================================================

def update_rho_value(change):
    rho_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{rho_slider.value:.2f}</div>'

def update_sigma_value(change):
    sigma_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{sigma_slider.value:.1f}</div>'

def update_N_value(change):
    N_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{N_slider.value}</div>'

def update_lag_value(change):
    lag_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{lag_slider.value}</div>'

rho_slider.observe(update_rho_value, names='value')
sigma_slider.observe(update_sigma_value, names='value')
N_slider.observe(update_N_value, names='value')
lag_slider.observe(update_lag_value, names='value')

# ============================================================
# CONTROL LABELS
# ============================================================

rho_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Correlation ρ:</div>')
sigma_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Std. deviation σ:</div>')
N_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Samples N:</div>')
lag_label = HTML('<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Maximum lag:</div>')

# ============================================================
# CONTROLS GRID
# ============================================================

controls_grid = GridBox(
    children=[
        rho_label, rho_slider, rho_value,
        sigma_label, sigma_slider, sigma_value,
        N_label, N_slider, N_value,
        lag_label, lag_slider, lag_value
    ],
    layout=Layout(
        width='390px',
        grid_template_columns='125px 165px 55px',
        grid_template_rows='34px 34px 34px 34px',
        grid_gap='4px 6px',
        align_items='center',
        overflow='hidden'
    )
)

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#12388c;
            margin-bottom:6px;
        ">
        Statistical Specifications
        </div>
        """),
        controls_grid
    ],
    layout=Layout(
        width='420px',
        padding='12px 14px',
        border='1px solid #d2d2d2',
        overflow='hidden'
    )
)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# MAIN INTERACTIVE FUNCTION
# ============================================================

def plot_ar1_design(rho=0.75, sigma=1.5, N=2000, max_lag=30):

    # --------------------------------------------------------
    # DESIGN PARAMETERS
    # --------------------------------------------------------

    alpha = rho
    beta = sigma * np.sqrt(1.0 - rho**2)

    # --------------------------------------------------------
    # WHITE-NOISE INPUT
    # --------------------------------------------------------

    rng = np.random.default_rng(24)
    w = rng.normal(0.0, 1.0, N)

    # --------------------------------------------------------
    # AR(1) PROCESS
    #
    # y[n] = alpha y[n-1] + beta w[n]
    # --------------------------------------------------------

    y = signal.lfilter([beta], [1.0, -alpha], w)

    transient = min(300, N // 10)
    y_ss = y[transient:]

    # ========================================================
    # THEORETICAL AUTOCORRELATION
    # ========================================================

    lags = np.arange(-max_lag, max_lag + 1)
    R_theory = sigma**2 * rho**np.abs(lags)

    # ========================================================
    # ESTIMATED AUTOCORRELATION
    # ========================================================

    y_centered = y_ss - np.mean(y_ss)
    full_corr = np.correlate(y_centered, y_centered, mode='full')
    center_index = len(full_corr) // 2
    selected_corr = full_corr[center_index - max_lag:center_index + max_lag + 1]
    normalization = len(y_centered) - np.abs(lags)
    R_est = selected_corr / normalization

    # ========================================================
    # THEORETICAL PSD
    # ========================================================

    omega = np.linspace(-np.pi, np.pi, 2048)
    S_theory = beta**2 / (1.0 + alpha**2 - 2.0 * alpha * np.cos(omega))

    # ========================================================
    # ESTIMATED PSD
    # ========================================================

    nperseg = min(512, len(y_ss))
    f_est, Pyy = signal.welch(y_ss, fs=2.0 * np.pi, window='hann', nperseg=nperseg, noverlap=nperseg // 2, return_onesided=False, scaling='density')

    f_est = np.fft.fftshift(f_est)
    Pyy = np.fft.fftshift(Pyy)
    Pyy = 2.0 * np.pi * Pyy

    # ========================================================
    # SINGLE FIGURE WITH THREE AXES
    # ========================================================

    fig = plt.figure(figsize=(10.4, 6.8))

    gs = fig.add_gridspec(
        2,
        2,
        height_ratios=[1.0, 1.05],
        hspace=0.48,
        wspace=0.30
    )

    ax1 = fig.add_subplot(gs[0, :])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[1, 1])

    # ========================================================
    # GRAPH 1:
    # PROCESS REALIZATION
    # ========================================================

    show_N = min(500, len(y_ss))

    ax1.plot(np.arange(show_N), y_ss[:show_N], linewidth=1.0)

    ax1.set_xlim(0, show_N - 1)

    ax1.set_xlabel('Time index n', fontsize=11)

    ax1.set_ylabel('y[n]', fontsize=11)

    ax1.set_title(f'Generated AR(1) Random Process, α = {alpha:.2f}, β = {beta:.3f}', fontsize=13, pad=9)

    ax1.tick_params(axis='both', labelsize=9)

    ax1.grid(True, linestyle=':', alpha=0.5)

    # ========================================================
    # GRAPH 2:
    # AUTOCORRELATION
    # ========================================================

    ax2.plot(lags, R_est, linewidth=1.8, label='Estimated autocorrelation')

    ax2.plot(lags, R_theory, linestyle='--', linewidth=2.0, label='Theoretical autocorrelation')

    ax2.axhline(0, linewidth=0.8)

    ax2.axvline(0, linewidth=0.8, linestyle=':')

    ax2.set_xlim(-max_lag, max_lag)

    ax2.set_xlabel('Lag m', fontsize=11)

    ax2.set_ylabel('Ryy[m]', fontsize=11)

    ax2.set_title('Output Autocorrelation', fontsize=13, pad=9)

    ax2.tick_params(axis='both', labelsize=9)

    ax2.grid(True, linestyle=':', alpha=0.5)

    ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=1, fontsize=8)

    # ========================================================
    # GRAPH 3:
    # PSD
    # ========================================================

    ax3.plot(omega, S_theory, linewidth=2.0, label='Theoretical PSD')

    ax3.plot(f_est, Pyy, linewidth=1.2, alpha=0.75, label='Estimated PSD')

    ax3.set_xlim(-np.pi, np.pi)

    ax3.set_xticks([-np.pi, -np.pi / 2.0, 0.0, np.pi / 2.0, np.pi])

    ax3.set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])

    ax3.set_xlabel('Angular frequency ω', fontsize=11)

    ax3.set_ylabel('PSD', fontsize=11)

    ax3.set_title('Output Power Spectral Density', fontsize=13, pad=9)

    ax3.tick_params(axis='both', labelsize=9)

    ax3.grid(True, linestyle=':', alpha=0.5)

    ax3.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=1, fontsize=8)

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(left=0.08, right=0.97, top=0.93, bottom=0.13)

    plt.show()

    plt.close(fig)

    # ========================================================
    # DESIGN RESULT
    # ========================================================

    R0_est = R_est[max_lag]

    if max_lag >= 1:
        R1_est = R_est[max_lag + 1]
    else:
        R1_est = np.nan

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.42;
        width:880px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Designed parameters:</b>
    α = ρ = {alpha:.3f},
    &nbsp;&nbsp;
    β = σ√(1−ρ²) = {beta:.4f}

    <br>

    <b>Target:</b>
    Rᵧᵧ[0] = {sigma**2:.4f},
    &nbsp;&nbsp;
    Rᵧᵧ[1] = {rho*sigma**2:.4f}

    <br>

    <b>Estimated:</b>
    Rᵧᵧ[0] ≈ {R0_est:.4f},
    &nbsp;&nbsp;
    Rᵧᵧ[1] ≈ {R1_est:.4f}

    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(
    plot_ar1_design,
    {
        'rho': rho_slider,
        'sigma': sigma_slider,
        'N': N_slider,
        'max_lag': lag_slider
    }
)

# ============================================================
# SHORT INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.42;
    width:1050px;
    padding:11px 15px;
    border:1px solid #c8dfce;
    background:#f8fcf9;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#197b35;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The parameter ρ directly determines the rate at which the autocorrelation decays with lag.
</div>

<div style="margin-bottom:4px;">
The parameter σ determines the required output variance, while β is automatically chosen so that Rᵧᵧ[0] = σ².
</div>

<div>
The estimated autocorrelation and PSD approach the theoretical curves as the number of generated samples increases.
</div>

</div>
""")

# ============================================================
# TOP AREA
# ============================================================

top_layout = HBox(
    [
        documentation,
        controls_card
    ],
    layout=Layout(
        width='1480px',
        align_items='flex-start',
        justify_content='flex-start',
        gap='20px'
    )
)

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        top_layout,
        output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

display(main_layout)